# M6 — Aula 01: tensores, a ponte entre NumPy e PyTorch

[Aula completa](../aulas/01-tensores-numpy-pytorch.md) · [Currículo M6](../README.md)

## Objetivo

Transportar cálculos do M5 para tensores sem alterar shapes, tipos ou dados inadvertidamente. Vamos conferir um forward pequeno em NumPy e PyTorch, sem autograd, `nn.Module`, treinamento ou GPU. Esses assuntos têm aulas próprias na grade.

## Ambiente

Python >= 3.10, NumPy >= 1.24, PyTorch >= 2.6. Referência executada: Python 3.12.14, NumPy 2.3.5 e PyTorch 2.6.0+cpu. CPU é suficiente. Não há downloads de dados, segredos ou serviços externos. `nbformat >= 5.7` é necessário apenas para validar o formato, não para os cálculos.

Em um ambiente virtual novo, instale NumPy e a distribuição PyTorch adequada à plataforma. Para reproduzir a referência em Linux/Windows CPU: `python -m pip install 'numpy==2.3.5'`, seguido de `python -m pip install 'torch==2.6.0' --index-url https://download.pytorch.org/whl/cpu`. Para outras plataformas, consulte as [instruções oficiais por versão](https://pytorch.org/get-started/previous-versions/). Não reinstale pacotes durante o laboratório; reinicie o kernel se alterar o ambiente.

Execute todas as células em ordem. O notebook versionado fica com outputs limpos depois de executar uma cópia de validação. Resultados de referência constam da aula. Versões mínimas são um contrato de dependências, não uma alegação de teste de todas as combinações.

### Resultado confirmado

A CE foi 1,109595136439 nas duas implementações; na fixture aleatória o erro máximo foi 8,88 × 10⁻¹⁶. A contraprova de broadcasting produziu MSE 1,333333 em vez de zero. Os nove grupos de auditoria passaram no ambiente CPU declarado. Esses resultados verificam operações locais, não um treinamento do P6.


In [ ]:
import sys
import warnings
import numpy as np
import torch

warnings.simplefilter("error")
np.seterr(divide="raise", over="raise", invalid="raise")
torch.set_num_threads(1)
SEED = 20260901
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
DEVICE = torch.device("cpu")
DTYPE = torch.float64
checks = {}
print({"python": sys.version.split()[0], "numpy": np.__version__,
       "torch": torch.__version__, "device": str(DEVICE)})

## 1. Shape, dtype e device

Um escalar tem shape `()`, um vetor `(D,)`, e uma matriz `(B,D)`. `ndim` conta eixos; não é o posto algébrico de uma matriz. Os elementos têm um dtype, e o tensor reside em um dispositivo. Inspecionar esses atributos evita inferir semântica apenas olhando números.

In [ ]:
scalar = torch.tensor(2.0, dtype=DTYPE, device=DEVICE)
vector = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE, device=DEVICE)
batch = torch.tensor([[1., 2., 3.], [4., 5., 6.]], dtype=DTYPE, device=DEVICE)
assert scalar.shape == torch.Size([]) and scalar.ndim == 0
assert vector.shape == (3,) and batch.shape == (2, 3)
assert batch.numel() == 6 and batch.dtype == torch.float64
assert batch.device.type == "cpu" and not batch.requires_grad
assert torch.tensor([1, 2]).dtype == torch.int64
assert torch.tensor([1.0, 2.0]).dtype == torch.get_default_dtype()
checks["atributos"] = True
print("Escalar/vetor/lote:", tuple(scalar.shape), tuple(vector.shape), tuple(batch.shape))

## 2. Cópia ou memória compartilhada?

`torch.tensor(array)` copia os dados. `torch.from_numpy(array)` compartilha o armazenamento de um array CPU compatível. `torch.as_tensor(array)` também pode compartilhar quando tipo e dispositivo permitem. Nos exemplos usamos arrays contíguos, graváveis e float64. Não escreva por um tensor criado a partir de array somente leitura.

A mutação é deliberada para mostrar o risco; em um pipeline real, decida primeiro quem pode alterar o dado.

In [ ]:
array = np.array([1., 2., 3.], dtype=np.float64)
shared = torch.from_numpy(array)
copied = torch.tensor(array)
adapted = torch.as_tensor(array)
array[0] = 10.0
assert shared[0].item() == adapted[0].item() == 10.0
assert copied[0].item() == 1.0
shared[1] = -2.0
assert array[1] == -2.0
independent = shared.clone()
shared[2] = 99.0
assert independent[2].item() == 3.0
assert np.shares_memory(array, shared.numpy())
checks["compartilhamento_copia"] = True
print("Array após mutações:", array.tolist(), "Cópia:", copied.tolist())

## 3. Converter tipo não significa sempre copiar

`.to(...)` retorna o próprio tensor se não precisar converter e `copy=False`, o padrão. Atribua o retorno; o método não muda o dtype do objeto original. Use `clone()` para armazenamento independente ou `to(..., copy=True)` quando quiser uma cópia explícita.

Ainda não há grafo nesta aula. `clone()` não deve ser confundido com desligar autograd: quando há grafo, ele preserva a relação diferenciável, assunto da Aula 04.

In [ ]:
original = torch.tensor([1., 2.], dtype=torch.float64)
same = original.to(dtype=torch.float64, device="cpu")
converted = original.to(dtype=torch.float32)
forced = original.to(copy=True)
assert same is original
assert converted.dtype == torch.float32 and original.dtype == torch.float64
forced[0] = 77.0
assert original[0].item() == 1.0
converted[1] = 88.0
assert original[1].item() == 2.0
checks["conversao_to"] = True
print("Mesmo objeto sem conversão:", same is original)

## 4. Precisão e tolerância

Float32 não representa todos os inteiros grandes: em torno de 2**24, somar 1 pode perder informação. Float64 preserva essa soma específica. Isso não torna float64 exato em geral nem necessário para todo treinamento.

`assert_close` verifica discrepâncias com tolerância absoluta e relativa: `abs(a-b) <= atol + rtol*abs(b)`. Escolha a tolerância conforme escala e precisão, depois investigue falhas; aumentar tolerâncias até passar não explica um erro.

In [ ]:
v32 = torch.tensor(2**24, dtype=torch.float32)
v64 = torch.tensor(2**24, dtype=torch.float64)
assert (v32 + 1).item() == 2**24
assert (v64 + 1).item() == 2**24 + 1
torch.testing.assert_close(v32.to(torch.float64), v64, rtol=0, atol=0)
checks["precisao"] = True
print("float32 + 1:", (v32 + 1).item(), "float64 + 1:", (v64 + 1).item())

## 5. Fixture afim resolvida à mão

Mantemos a convenção do P5: linhas são exemplos, `X` tem shape `(B,D)` e `W`, `(D,H)`. O produto é `X @ W`; `X * W` significa multiplicação elemento a elemento e não substitui o produto matricial.

Para a primeira unidade da primeira linha: `1*0.5 + 2*1 + (-1)*(-0.5) + 0.25 = 3.25`. O bias `(H,)` é somado a todas as linhas por broadcasting. A matriz de pesos usada por `nn.Linear` será orientada diferentemente, assunto da Aula 07.

In [ ]:
Xnp = np.array([[1., 2., -1.], [0., -1., 3.]], dtype=np.float64)
Wnp = np.array([[0.5, -1.], [1., 0.], [-0.5, 2.]], dtype=np.float64)
bnp = np.array([0.25, -0.5], dtype=np.float64)
X = torch.tensor(Xnp, dtype=DTYPE, device=DEVICE)
W = torch.tensor(Wnp, dtype=DTYPE, device=DEVICE)
b = torch.tensor(bnp, dtype=DTYPE, device=DEVICE)
Znp = Xnp @ Wnp + bnp
Z = X @ W + b
expected = torch.tensor([[3.25, -3.5], [-2.25, 5.5]], dtype=DTYPE)
assert X.shape == (2, 3) and W.shape == (3, 2) and b.shape == (2,)
torch.testing.assert_close(Z, expected, atol=0, rtol=0)
np.testing.assert_allclose(Z.numpy(), Znp, atol=1e-12, rtol=1e-12)
checks["afim"] = True
print("Z =", Z.tolist())

## 6. Forward de uma pequena MLP, sem camadas prontas

Aplicamos ReLU, segunda transformação afim e log-softmax estável. A CE média é calculada diretamente para conferir o forward. Backward, `nn.Module` e APIs de loss serão introduzidos gradualmente nas aulas seguintes. Rótulos são índices int64; features e pesos são float64.

In [ ]:
W2np = np.array([[0.2, -0.3], [0.4, 0.1]], dtype=np.float64)
b2np = np.array([0.1, -0.2], dtype=np.float64)
labels = torch.tensor([0, 1], dtype=torch.int64)
A = torch.relu(Z)
logits = A @ torch.tensor(W2np) + torch.tensor(b2np)
shifted = logits - logits.max(dim=1, keepdim=True).values
logp = shifted - torch.log(torch.exp(shifted).sum(dim=1, keepdim=True))
loss = -logp[torch.arange(len(labels)), labels].mean()

Anp = np.maximum(Znp, 0)
logits_np = Anp @ W2np + b2np
shifted_np = logits_np - logits_np.max(axis=1, keepdims=True)
logp_np = shifted_np - np.log(np.exp(shifted_np).sum(axis=1, keepdims=True))
loss_np = -logp_np[np.arange(2), np.array([0, 1])].mean()
assert logits.shape == (2, 2) and loss.ndim == 0
assert torch.isfinite(logp).all().item()
np.testing.assert_allclose(logits.numpy(), logits_np, atol=1e-12, rtol=1e-12)
assert abs(loss.item() - loss_np) < 1e-12
checks["forward_mlp"] = True
print("Logits:", logits.tolist())
print(f"CE torch={loss.item():.12f}; numpy={loss_np:.12f}")

## 7. Um erro silencioso de broadcasting

O exemplo de regressão a seguir é uma contraprova, não treinamento. `(3,) - (3,1)` é válido e gera `(3,3)`, comparando cada predição com todos os alvos. Mesmo com números iguais, a MSE fica errada. Para uma predição por exemplo, imponha shapes iguais antes da subtração. Na Aula 02 estudaremos as regras de eixos em detalhe.

In [ ]:
pred = torch.tensor([1., 2., 3.], dtype=DTYPE)
target_column = torch.tensor([[1.], [2.], [3.]], dtype=DTYPE)
wrong_residual = pred - target_column
wrong_mse = (wrong_residual ** 2).mean()
target = target_column.squeeze(1)
assert pred.shape == target.shape
right_mse = ((pred - target) ** 2).mean()
assert wrong_residual.shape == (3, 3)
assert abs(wrong_mse.item() - 4/3) < 1e-12 and right_mse.item() == 0.0
checks["contraprova_broadcast"] = True
print(f"MSE incorreta={wrong_mse.item():.6f}; correta={right_mse.item():.6f}")

## 8. Um erro explícito de dtype

Multiplicação matricial densa exige dtypes compatíveis. Capturamos deliberadamente a exceção esperada e corrigimos os tipos de forma explícita; nenhum aviso é silenciado. Em produção, não use um `except` amplo para esconder defeitos do pipeline.

In [ ]:
failed = False
try:
    X.to(torch.float32) @ W
except RuntimeError:
    failed = True
assert failed, "Esperávamos incompatibilidade float32/float64"
fixed = X.to(torch.float32) @ W.to(torch.float32)
assert fixed.dtype == torch.float32
torch.testing.assert_close(fixed.to(torch.float64), X @ W, atol=1e-6, rtol=1e-6)
checks["erro_dtype_controlado"] = True
print("Erro esperado capturado; produto com dtype coerente aprovado")

## 9. Seed controla um gerador, não a equivalência entre bibliotecas

Geradores PyTorch reiniciados com a mesma seed reproduzem a sequência neste ambiente. NumPy possui gerador próprio. Para testar a migração de uma operação, gere uma fixture uma vez e transporte os mesmos valores; pedir duas inicializações aleatórias com o mesmo número de seed não é esse contrato.

Usamos uma fixture sintética independente de MNIST. Ela não replica o treinamento do P5, nem mede generalização: verifica operações locais. Não há split ou pré-processamento aprendido neste laboratório.

In [ ]:
g1 = torch.Generator(device="cpu").manual_seed(SEED)
g2 = torch.Generator(device="cpu").manual_seed(SEED)
a1 = torch.randn(8, generator=g1, dtype=DTYPE)
a2 = torch.randn(8, generator=g2, dtype=DTYPE)
assert torch.equal(a1, a2)
X_random = rng.normal(size=(7, 5))
W_random = rng.normal(size=(5, 4))
b_random = rng.normal(size=(4,))
expected_random = np.maximum(X_random @ W_random + b_random, 0)
actual_random = torch.relu(torch.tensor(X_random) @ torch.tensor(W_random) + torch.tensor(b_random))
max_error = np.max(np.abs(actual_random.numpy() - expected_random))
np.testing.assert_allclose(actual_random.numpy(), expected_random, atol=1e-12, rtol=1e-12)
checks["seed_fixture"] = True
print(f"Erro máximo na fixture aleatória compartilhada: {max_error:.3e}")

## 10. Auditoria final e interpretação

NumPy só recebe aqui tensores CPU sem gradientes. O caso com autograd exigirá discutir `detach`, sem confundir essa operação com cópia de armazenamento. Os valores numéricos são verificados com tolerâncias; igualdade neste ambiente não promete identidade entre CPU, GPU ou versões.

Um teste que passa confirma seu escopo: o forward confere, mas nada foi afirmado sobre backward, desempenho, calibração ou qualidade de um modelo treinado.

In [ ]:
assert len(checks) == 9 and all(checks.values())
assert not logits.requires_grad and not loss.requires_grad
results = {"checks": len(checks), "affine": Z.tolist(), "logits": logits.tolist(),
           "ce": loss.item(), "ce_numpy_error": float(abs(loss.item()-loss_np)),
           "random_max_error": float(max_error), "wrong_mse": wrong_mse.item(),
           "right_mse": right_mse.item(), "torch": torch.__version__}
print(results)
print("9/9 grupos de auditoria aprovados")

## Exercícios com respostas

1. `from_numpy` deve ser usado para obter independência do array? **Não:** use uma cópia explícita; o laboratório mostra alteração nos dois sentidos.
2. Por que dtype deve acompanhar a fixture? **Porque** conversão de precisão pode mudar números, e a operação matricial precisa de tipos compatíveis.
3. O produto `(B,D) @ (D,H)` retorna qual shape? **`(B,H)`**; o eixo D é contraído.
4. Por que a MSE incorreta é finita? **Porque** broadcasting produziu uma operação válida sobre todos os pares; finitude não valida a semântica.
5. O forward equivalente prova o P6? **Não:** ainda faltam gradientes, laço, estado, dados e avaliação controlada.

Próxima aula: **02 — Eixos, indexação, broadcasting e layout**, conforme o currículo. Autograd começa na Aula 03.

### Fontes técnicas

- [PyTorch: tutorial oficial de tensores](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html).
- [PyTorch 2.6: from_numpy](https://docs.pytorch.org/docs/2.6/generated/torch.from_numpy.html).
- [PyTorch 2.14: tensor](https://docs.pytorch.org/docs/2.14/generated/torch.tensor.html).
- [PyTorch 2.14: Tensor.to](https://docs.pytorch.org/docs/2.14/generated/torch.Tensor.to.html).
- [PyTorch 2.6: reprodutibilidade](https://docs.pytorch.org/docs/2.6/notes/randomness.html).

Consulta em 9 de setembro de 2026. As páginas de tensor e Tensor.to consultadas são da documentação 2.14; os comportamentos usados foram exercitados em 2.6.0. O tutorial corrente pode mudar. Não tratamos PyTorch 2.6 como a versão mais recente.